# 🎯 Auto-interp · Targeted run for missing circuit features

Companion to [04b](./04b_autointerp_qwen36_27b_papergrade.ipynb). The main run labeled the top-500 *corpus-frequent* features per layer; circuit features are *scenario-active* and only ~8/44 overlapped. This notebook fills the gap.

**Inputs**: hardcoded list of 36 (layer, feature_id) pairs from the 4 Circuit Canvas scenarios that lacked labels in the v0.1.0 catalog.

**Pipeline**:
1. Load Qwen3.6-27B + L11 + L31 SAEs (frozen) — L55 not needed (no circuits use it currently)
2. Pass B targeted: stream 3M tokens, capture top-20 contexts ONLY for the 36 specified features
3. Auto-interp via Claude Opus 4.7 (OpenRouter)
4. Merge new labels into the existing `feature_catalog.json` on HF SAE repo (additive — does not touch the 1500 from 04b)

**Cost**: ~15 min GPU + ~$1 OpenRouter (36 features × ~750 tok input × Opus 4.7).

After this runs, every node in the 4 circuit scenarios will have a label (or explicit refusal/polysemantic flag).

In [ ]:
!pip install -q -U transformers accelerate safetensors huggingface_hub datasets tqdm requests
import torch, transformers
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)

## 1. Config + targeted feature list

In [ ]:
HF_SAE_REPO   = 'caiovicentino1/qwen36-27b-sae-papergrade'
HF_BASE_MODEL = 'Qwen/Qwen3.6-27B'
LAYERS        = [11, 31]   # L55 not needed — current circuits use L11→L31
D_MODEL       = 5120
D_SAE         = 65_536
K             = 128

# Targeted features that appear in the 4 Circuit Canvas scenarios but were
# NOT in the top-500 corpus-frequent labels from 04b.
TARGETED = {
    11: [
        # ioi up
        39362, 38927, 51405, 54276, 4332, 20842,
        # math up
        50810, 21195, 20507, 60298, 30629,
        # refusal up
        36804, 55991, 32010, 41136,
        # medical up
        64163, 17728, 2541,
    ],
    31: [
        # ioi down
        16060, 32631, 61135, 51925, 52480, 45703,
        # math down
        10605, 34845, 19535, 36089,
        # refusal down
        21273, 43943, 37423, 49404,
        # medical down
        37602, 46598, 39620, 15905,
    ],
}

CORPUS_TOKENS  = 3_000_000   # smaller corpus — only tracking 36 features
BATCH_TOKENS   = 4_096
SEQ_LEN        = 1_024
CONTEXT_HALF_WIDTH = 16
TOP_CONTEXTS_PER_FEATURE = 20

OPENROUTER_MODEL = 'anthropic/claude-opus-4.7'
OPENROUTER_URL   = 'https://openrouter.ai/api/v1/chat/completions'
API_MAX_TOKENS   = 60
API_TEMPERATURE  = 0.0
API_RPM_LIMIT    = 30

import os, math, json, time, random, requests
random.seed(0); torch.manual_seed(0)

CACHE_DIR = '/content/drive/MyDrive/autointerp_qwen36_27b'
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    os.makedirs(CACHE_DIR, exist_ok=True)
except Exception:
    CACHE_DIR = '/tmp/autointerp_qwen36_27b'
    os.makedirs(CACHE_DIR, exist_ok=True)

n_targeted = sum(len(v) for v in TARGETED.values())
print(f'targeted features: {n_targeted} ({len(TARGETED[11])} at L11, {len(TARGETED[31])} at L31)')
print(f'cache: {CACHE_DIR}')

## 2. Auth (HF + OpenRouter) + smoke test

In [ ]:
from huggingface_hub import login, hf_hub_download
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    assert OPENROUTER_API_KEY
except Exception:
    login()
    OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY') or input('OPENROUTER_API_KEY: ')

r = requests.post(
    OPENROUTER_URL,
    headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}', 'Content-Type': 'application/json'},
    json={'model': OPENROUTER_MODEL, 'messages': [{'role': 'user', 'content': 'Reply with exactly: OK'}],
          'max_tokens': 5, 'temperature': 0.0},
    timeout=30,
)
assert r.status_code == 200, r.text
print('OpenRouter OK ·', OPENROUTER_MODEL)

## 3. Load base model + L11/L31 SAEs (skip L55 — not needed)

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from safetensors.torch import load_file
import torch.nn.functional as F

device = 'cuda'
tok = AutoTokenizer.from_pretrained(HF_BASE_MODEL, trust_remote_code=True)
model = AutoModelForImageTextToText.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map='cuda',
    trust_remote_code=True,
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

class TopKSAE(torch.nn.Module):
    def __init__(self, sd, k):
        super().__init__()
        self.W_enc = torch.nn.Parameter(sd['W_enc'].to(torch.bfloat16), requires_grad=False)
        self.b_enc = torch.nn.Parameter(sd['b_enc'].to(torch.bfloat16), requires_grad=False)
        self.W_dec = torch.nn.Parameter(sd['W_dec'].to(torch.bfloat16), requires_grad=False)
        self.b_dec = torch.nn.Parameter(sd['b_dec'].to(torch.bfloat16), requires_grad=False)
        self.k = k
    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc
        vals, idx = pre.topk(self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre)
        z.scatter_(-1, idx, vals)
        return z

saes = {}
for layer in LAYERS:
    path = hf_hub_download(HF_SAE_REPO, f'sae_L{layer}_latest.safetensors')
    saes[layer] = TopKSAE(load_file(path), K).to(device).eval()
    print(f'  ✓ SAE L{layer}')

layer_mods = {layer: model.model.language_model.layers[layer] for layer in LAYERS}
print(f'\nready · vram free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## 4. Pass B (targeted) — capture top-20 contexts per missing feature

In [ ]:
from datasets import load_dataset
from tqdm.auto import tqdm
import heapq

_captured = {}
def capture_hook(key):
    def h(mod, inp, out):
        _captured[key] = out[0] if isinstance(out, tuple) else out
        return out
    return h

def stream_batches(total_tokens=CORPUS_TOKENS):
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT',
                       split='train', streaming=True)
    buf, seen, batch_id = [], 0, 0
    for ex in ds:
        ids = tok(ex['text'], truncation=True, max_length=SEQ_LEN)['input_ids']
        buf.extend(ids)
        while len(buf) >= BATCH_TOKENS:
            chunk = buf[:BATCH_TOKENS]; buf = buf[BATCH_TOKENS:]
            seen += len(chunk)
            B = BATCH_TOKENS // SEQ_LEN
            arr = torch.tensor(chunk, device=device).reshape(B, SEQ_LEN)
            yield arr, batch_id
            batch_id += 1
            if seen >= total_tokens:
                return

ctx_heap = {layer: {f: [] for f in TARGETED[layer]} for layer in LAYERS}

hs = []
for layer in LAYERS:
    hs.append(layer_mods[layer].register_forward_hook(capture_hook(layer)))

n_steps = CORPUS_TOKENS // BATCH_TOKENS
with torch.no_grad():
    for batch, batch_id in tqdm(stream_batches(), total=n_steps, desc='Pass B targeted'):
        _ = model(batch)
        B_, T_ = batch.shape
        for layer in LAYERS:
            resid = _captured[layer]
            flat = resid.reshape(-1, D_MODEL)
            z = saes[layer].encode(flat.to(torch.bfloat16))
            z = z.view(B_, T_, D_SAE)
            tracked = TARGETED[layer]
            sub_z = z[:, :, tracked]   # (B, T, n_targeted)
            mask = sub_z > 0
            if not mask.any():
                continue
            nz_b, nz_t, nz_i = mask.nonzero(as_tuple=True)
            vals = sub_z[nz_b, nz_t, nz_i].float().cpu().tolist()
            for v, b, t, i in zip(vals, nz_b.cpu().tolist(),
                                       nz_t.cpu().tolist(),
                                       nz_i.cpu().tolist()):
                feat = tracked[i]
                heap = ctx_heap[layer][feat]
                if len(heap) < TOP_CONTEXTS_PER_FEATURE:
                    heapq.heappush(heap, (v, batch_id, b, t))
                elif v > heap[0][0]:
                    heapq.heapreplace(heap, (v, batch_id, b, t))
        torch.save(batch.cpu(), os.path.join(CACHE_DIR, f'_tgt_batch_{batch_id:06d}.pt'))

for h in hs:
    h.remove()

# Reify
def reify(batch_id, b, t):
    arr = torch.load(os.path.join(CACHE_DIR, f'_tgt_batch_{batch_id:06d}.pt'))
    seq = arr[b].tolist()
    lo = max(0, t - CONTEXT_HALF_WIDTH)
    hi = min(len(seq), t + CONTEXT_HALF_WIDTH + 1)
    pre  = tok.decode(seq[lo:t],   skip_special_tokens=True)
    fire = tok.decode([seq[t]],    skip_special_tokens=True)
    post = tok.decode(seq[t+1:hi], skip_special_tokens=True)
    return pre, fire, post

contexts = {layer: {} for layer in LAYERS}
for layer in LAYERS:
    for feat, heap in ctx_heap[layer].items():
        items = sorted(heap, key=lambda x: -x[0])
        contexts[layer][feat] = [
            {'pre': p, 'fire': f, 'post': po, 'act': float(v)}
            for v, batch_id, b, t in items
            for (p, f, po) in [reify(batch_id, b, t)]
        ]
    sparse = sum(1 for f in TARGETED[layer] if len(contexts[layer][f]) < TOP_CONTEXTS_PER_FEATURE)
    if sparse:
        print(f'  L{layer}: {sparse}/{len(TARGETED[layer])} features had <{TOP_CONTEXTS_PER_FEATURE} contexts')

import glob
for p in glob.glob(os.path.join(CACHE_DIR, '_tgt_batch_*.pt')):
    os.remove(p)
print('  ✓ contexts captured · cache cleaned')

## 5. Auto-interp loop (Opus 4.7) — same prompt as 04b

In [ ]:
PROMPT_TEMPLATE = '''You are analyzing a feature in a Sparse Autoencoder trained on the residual stream of Qwen3.6-27B at layer {layer}.

Below are {n} token contexts where this feature activates strongly. Each context shows a window of text around the firing token. The token where the feature actually fires is marked with [[double brackets]].

{contexts_block}

Based on these contexts, what concept does this feature detect? Look for syntactic patterns, semantic categories, named entities, factual associations, or stylistic features. Respond with a SHORT label (3-8 words) that captures what unifies the activations.

If the feature appears polysemantic (firing on multiple unrelated concepts), respond with "polysemantic: X / Y" naming the two strongest patterns.

Reply with ONLY the label — no quotes, no preamble, no period.
'''

def format_contexts(ctxs):
    lines = []
    for i, c in enumerate(ctxs, 1):
        text = (c['pre'] + '[[' + c['fire'] + ']]' + c['post']).replace('\n', ' ')
        lines.append(f'{i}. "{text.strip()}"  (act={c["act"]:.2f})')
    return '\n'.join(lines)

def call_claude(prompt, retries=4):
    for attempt in range(retries):
        try:
            r = requests.post(
                OPENROUTER_URL,
                headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}', 'Content-Type': 'application/json'},
                json={'model': OPENROUTER_MODEL, 'messages': [{'role': 'user', 'content': prompt}],
                      'max_tokens': API_MAX_TOKENS, 'temperature': API_TEMPERATURE},
                timeout=60,
            )
            if r.status_code == 200:
                data = r.json()
                content = data['choices'][0]['message'].get('content')
                usage = data.get('usage', {})
                if not content:
                    fr = data['choices'][0].get('finish_reason', '?')
                    return f'(refused: finish={fr})', usage
                return content.strip().strip('"').strip("'").strip('.'), usage
            elif r.status_code == 429:
                time.sleep(2 ** attempt * 5)
            else:
                print(f'    http {r.status_code}: {r.text[:200]}')
                time.sleep(2 ** attempt)
        except Exception as e:
            print(f'    {e}; retry {attempt}')
            time.sleep(2 ** attempt)
    return None, {}

new_labels = {layer: {} for layer in LAYERS}
min_dt = 60.0 / API_RPM_LIMIT
last = 0.0
tot_in = tot_out = 0

for layer in LAYERS:
    print(f'\n=== L{layer} · {len(TARGETED[layer])} features ===')
    for feat in tqdm(TARGETED[layer], desc=f'L{layer}'):
        ctxs = contexts[layer][feat]
        if not ctxs:
            new_labels[layer][feat] = '(no activations in 3M tokens)'
            continue
        prompt = PROMPT_TEMPLATE.format(layer=layer, n=len(ctxs), contexts_block=format_contexts(ctxs))
        dt = time.time() - last
        if dt < min_dt:
            time.sleep(min_dt - dt)
        last = time.time()
        label, usage = call_claude(prompt)
        if label is None:
            label = '(api failed)'
        new_labels[layer][feat] = label
        tot_in  += usage.get('prompt_tokens', 0)
        tot_out += usage.get('completion_tokens', 0)

cost = tot_in/1e6*15 + tot_out/1e6*75
print(f'\ntokens · in={tot_in:,} · out={tot_out:,}')
print(f'cost  · ${cost:.2f}')

for layer in LAYERS:
    print(f'\n=== L{layer} new labels ===')
    for feat, label in new_labels[layer].items():
        print(f'  L{layer}/f{feat:5d}  {label}')

## 6. Merge into existing feature_catalog.json + re-upload

Pull the existing catalog from HF, add new labels (without overwriting existing), push back.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

existing = json.load(open(hf_hub_download(HF_SAE_REPO, 'feature_catalog.json')))
before_count = sum(len(v) for v in existing['labels'].values())
print(f'existing catalog: {before_count} labels')

for layer in LAYERS:
    layer_key = str(layer)
    if layer_key not in existing['labels']:
        existing['labels'][layer_key] = {}
    for feat, label in new_labels[layer].items():
        existing['labels'][layer_key][str(feat)] = label

after_count = sum(len(v) for v in existing['labels'].values())
added = after_count - before_count
print(f'after merge:      {after_count} labels (+{added} new from this run)')

from datetime import datetime, timezone
existing['version'] = 'v0.1.1'
existing['timestamp'] = datetime.now(timezone.utc).isoformat()
existing['notes'] = 'v0.1.1: added targeted labels for 36 features that appear in Circuit Canvas scenarios but were missed by v0.1.0 corpus-frequent top-500.'

merged_path = os.path.join(CACHE_DIR, 'feature_catalog.json')
with open(merged_path, 'w') as f:
    json.dump(existing, f, indent=2)

api = HfApi()
api.upload_file(
    path_or_fileobj=merged_path,
    path_in_repo='feature_catalog.json',
    repo_id=HF_SAE_REPO,
    commit_message=f'feature_catalog v0.1.1: +{added} targeted circuit features (Claude Opus 4.7)',
)
print(f'\n✓ merged catalog uploaded → https://huggingface.co/{HF_SAE_REPO}/blob/main/feature_catalog.json')